In [1]:
from instances.instances import generate_instances

instances = generate_instances(filename="cost50.pkl", instance_count=2000, cities=range(5, 51), seed=42)

In [2]:
from data.generation import generate_train_data
from data.adapters.input.basic import BasicInputAdapter
from data.adapters.output.cost import CostOutputAdapter

input_config = (BasicInputAdapter, 50)
output_config = (CostOutputAdapter, 50)

generate_train_data(
    instance_file="cost50.pkl", 
    data_filename="train_data.h5", 
    input_adapter_config=input_config, 
    output_adapter_config=output_config,
    num_workers=12,
    only_initial_state=True
)

Datos guardados en: /home/oscar/Escritorio/TSP-Framework/data/train_data.h5 (Tamaño: 2000)


In [3]:
from data.preprocessing import load_dataset, split_dataset

train_file, val_file = split_dataset("train_data.h5", 1600)

train_dataset = load_dataset(train_file)
val_dataset = load_dataset(val_file)

Cargando dataset original: /home/oscar/Escritorio/TSP-Framework/data/train_data.h5
Guardando Train puro (1600 muestras) en: train_data_train.h5
Guardando Test puro (400 muestras) en: train_data_test.h5
Dataset train_data_train.h5 cargado con 1600 muestras.
Dataset train_data_test.h5 cargado con 400 muestras.


In [4]:
from models.cost import TSPCostTransformer

# Parámetros del modelo
input_dim = 2
embed_dim = 64
num_heads = 4
num_encoder_layers = 2
dropout = 0.1

# Crear modelo
model = TSPCostTransformer(
    input_dim=input_dim,
    embed_dim=embed_dim,
    num_heads=num_heads,
    num_encoder_layers=num_encoder_layers,
    dropout_rate=dropout,
)

In [5]:
from training.sl import sl_train, LRConfig
from training.metrics import MSE, MAE

model = sl_train(
    model=model,
    epochs=10,
    train_set=train_dataset,
    val_set=val_dataset,
    batch_size=64,
    lr_config=LRConfig(value=1e-4),
    weight_decay=1e-5,
    loss_fn=MSE(),
    patience=10,
    metrics=[MAE()],
    metrics_filename="metrics.txt",
    seed=42
)

** Usando dispositivo: cpu

Epoch 1/10
    Train MSE: 17.1816 | Val MSE: 15.8415
    MAE: 3.8320
Epoch 2/10
    Train MSE: 14.2356 | Val MSE: 13.7855
    MAE: 3.5537
Epoch 3/10
    Train MSE: 12.4223 | Val MSE: 11.7662
    MAE: 3.2571
Epoch 4/10
    Train MSE: 10.3707 | Val MSE: 9.6260
    MAE: 2.9101
Epoch 5/10
    Train MSE: 8.3662 | Val MSE: 7.5269
    MAE: 2.5288
Epoch 6/10
    Train MSE: 6.3857 | Val MSE: 5.5779
    MAE: 2.1234
Epoch 7/10
    Train MSE: 4.6437 | Val MSE: 3.9098
    MAE: 1.7245
Epoch 8/10
    Train MSE: 3.1546 | Val MSE: 2.6446
    MAE: 1.3940
Epoch 9/10
    Train MSE: 2.2073 | Val MSE: 1.8357
    MAE: 1.1641
Epoch 10/10
    Train MSE: 1.6167 | Val MSE: 1.4133
    MAE: 1.0189

** Historial de entrenamiento guardado en: /home/oscar/Escritorio/TSP-Framework/experiments/metrics.txt
** Mejor modelo restaurado (Época 10): MSE = 1.4133
